# 03 - Transformación de Datos

Este notebook aplica transformaciones al dataset:
- Estandarizar unidades de medida
- Categorizar productos
- Analizar y tratar valores nulos
- Imputación condicional según porcentaje de NaN

## Importar librerías y módulos

In [1]:
import pandas as pd
from src.transform import (
    standardize_measurement_units,
    add_category,
    get_null_statistics,
    impute_by_null_threshold,
    calculate_average_price,
    remove_high_null_products,
    get_anomaly_report
)

In [2]:
# Cargar datos limpios del paso anterior
df = pd.read_csv('../data/processed/data_limpia.csv')
print(f"Datos cargados: {df.shape}")

Datos cargados: (252, 142)


## 1. Estandarizar unidades de medida

In [3]:
print("Unidades de medida ANTES:")
print(df['unidad_de_medida'].unique())

Unidades de medida ANTES:
<StringArray>
[               'Galón 128 Oz',                 'Galón 64 Oz',
               'Botella 24 Oz',               'Botella 16 Oz',
                  'Lata 16 Oz',              'Botella 250 Ml',
                    '10 Libra',                   '10 Libras',
                       'Libra',                  '800 gramos',
                   '800 gramo',                   '400 gramo',
               'Cartón 30 Uds',                   '350 gramo',
                   '650 gramo',                     '5 Libra',
                 '28.35 gramo',                 '453.6 gramo',
                       '26 G.',                       '14 Oz',
                   '907 gramo',           'Funda 10 Unidades',
       'Funda Viga 1265 gramo',       'Funda Viga 1375 gramo',
     'Viga Familiar 850 gramo',                'Viga Mediana',
      'Viga Mediana 550 gramo',       'Viga Mediana 567gramo',
        'Viga Grande 567gramo',       'Viga Grande 525 gramo',
               

In [4]:
df = standardize_measurement_units(df)
print("\nUnidades de medida DESPUÉS:")
print(df['unidad_de_medida'].unique())


Unidades de medida DESPUÉS:
<StringArray>
[               'Galón 128 Oz',                 'Galón 64 Oz',
               'Botella 24 Oz',               'Botella 16 Oz',
                  'Lata 16 Oz',              'Botella 250 Ml',
                    '10 Libra',                       'Libra',
                   '800 Gramo',                   '800 gramo',
                   '400 gramo',               'Cartón 30 Uds',
                   '350 gramo',                   '650 gramo',
                     '5 Libra',                 '28.35 gramo',
                 '453.6 gramo',                       '26 G.',
                       '14 Oz',                   '907 gramo',
             'Funda 10 Unidad',       'Funda Viga 1265 gramo',
       'Funda Viga 1375 gramo',     'Viga Familiar 850 gramo',
                'Viga Mediana',      'Viga Mediana 550 gramo',
       'Viga Mediana 567gramo',        'Viga Grande 567gramo',
       'Viga Grande 525 gramo',                   '900 gramo',
            

## 2. Categorizar productos

In [5]:
df = add_category(df)
print(f"Categorías únicas: {df['categoria'].nunique()}")
print("\nDistribución por categoría:")
print(df['categoria'].value_counts())

Categorías únicas: 25

Distribución por categoría:
categoria
Leche en polvo    49
Vegetales         25
Aceite            24
Legumbres         18
Frutas            17
Leche liquida     13
Pollo             12
Pan               12
Arroz              9
Otro               9
Pescado            7
Lacteos            7
Carne res          6
Carne cerdo        6
Vinagre            6
Pasta              5
Sopita             5
Huevos             4
Embutidos          4
Harina             4
Avena              2
Azúcar             2
Café               2
Chocolate          2
Sal                2
Name: count, dtype: int64


C:\Users\Mariana\Documents\Git\Practicas\Indices_precio_globales\src\transform.py:97: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['categoria'] = df['nombre'].apply(asignar_categoria)


## 3. Analizar valores nulos

In [6]:
# Obtener columnas de fecha
cols_fecha = [col for col in df.select_dtypes(include='number').columns if col != 'orden']
print(f"Columnas de precio: {len(cols_fecha)}")

Columnas de precio: 139


In [7]:
# Obtener estadísticas de valores nulos
null_stats = get_null_statistics(df, cols_fecha)
print("\nProductos con más valores nulos (top 15):")
print(null_stats.head(15).to_string())


Productos con más valores nulos (top 15):
                                  nombre       categoria  null_count  null_percentage  valores_validos
168     Leche en polvo Kanny Instantánea  Leche en polvo         137            98.56                2
183             Leche en polvo Dos Pinos  Leche en polvo         137            98.56                2
133                         Sopita Maggi          Sopita         137            98.56                2
181             Leche en polvo Dos Pinos  Leche en polvo         136            97.84                3
116             Pan Integral Natures Own             Pan         125            89.93               14
165      Leche en polvo Nido Crecimiento  Leche en polvo         105            75.54               34
128                      Vinagre Barcelo         Vinagre         104            74.82               35
197                  Leche Líquida Nutra   Leche liquida         103            74.10               36
194           Leche Entera Líq

In [8]:
# Visualizar distribución de NaN
print("\nDistribución de estrategias de imputación:")
print(f"  0-15% NaN (Interpolación): {(null_stats['null_percentage'] < 15).sum()} productos")
print(f"  15-30% NaN (Forward/Backward fill): {((null_stats['null_percentage'] >= 15) & (null_stats['null_percentage'] < 30)).sum()} productos")
print(f"  >30% NaN (Media por categoría): {(null_stats['null_percentage'] >= 30).sum()} productos")


Distribución de estrategias de imputación:
  0-15% NaN (Interpolación): 215 productos
  15-30% NaN (Forward/Backward fill): 6 productos
  >30% NaN (Media por categoría): 31 productos


## 4. Aplicar imputación condicional

In [9]:
# 1. Definir umbrales
Bajo_Pct = 15
Alto_Pct = 30

print("--- REPORTE INICIAL DE CALIDAD ---")
stats_inicial = get_null_statistics(df, cols_fecha)
print(f"NaN totales: {df[cols_fecha].isna().sum().sum()}")
print(f"Productos con mucha falta de datos (>30%): {len(stats_inicial[stats_inicial['null_percentage'] > Alto_Pct])}")


#Guardar copia antes de inputar
df_antes_imputacion = df.copy()

# 2. Aplicar Imputación (Rellenar lo que se puede)
df= impute_by_null_threshold(df, cols_fecha, threshold_low=Bajo_Pct, threshold_high=Alto_Pct)

# 3. Eliminar lo que no tiene remedio (Los que tienen >30% de nulos)
df = remove_high_null_products(df, cols_fecha, max_null_pct=Alto_Pct)

print("\n--- VALIDACIÓN FINAL ---")
nan_final = df[cols_fecha].isna().sum().sum()
print(f"Total de NaN al final de todo el proceso: {nan_final}")
if nan_final == 0:
    print("¡Éxito! Los datos están limpios para el análisis.")
else:
    print(f"Aún quedan {nan_final} NaNs. Revisa si hay columnas vacías o casos extremos.")

--- REPORTE INICIAL DE CALIDAD ---
NaN totales: 3073
Productos con mucha falta de datos (>30%): 31

ELIMINACIÓN DE PRODUCTOS CON >30% NaN
Productos eliminados: 31
Productos restantes: 221

Productos eliminados:
                             nombre      categoria
               Aceite Mazola canola         Aceite
               Aceite de oliva Goya         Aceite
                Arroz selecto Chino          Arroz
           Arroz selecto San Rafael          Arroz
                    Gandules Verdes      Legumbres
                  Huevos Don Chichi         Huevos
               Espaguetis del César          Pasta
       Pollo procesado fresco Yaque          Pollo
Pollo procesado congelado Uni Pollo          Pollo
    Pollo procesado congelado Yaque          Pollo
       Muslo pollo entero Uni Pollo          Pollo
           Muslo pollo entero Yaque          Pollo
           Pan Integral Natures Own            Pan
                    Vinagre Barcelo        Vinagre
                       S

## Detenccion y exclusion de

In [ ]:
from pathlib import Path

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

anom_thresh = 40

reporte_anomalias = get_anomaly_report(df, cols_fecha, threshold_pct=anom_thresh)
df_anom = reporte_anomalias['anomalias_datos']

print(f"Total de anomalías detectadas: {reporte_anomalias['total_anomalias']}")
print(f"Productos afectados: {reporte_anomalias['productos_afectados']}")

if not df_anom.empty:
    df_anom.to_csv(output_dir / 'anomalias_detectadas.csv', index=False, encoding='utf-8-sig')
    
    df_anom['id_unico'] = df_anom['nombre'].astype(str) + " | " + df_anom['unidad_de_medida'].astype(str)
    ids_con_error = df_anom['id_unico'].unique()
    
    df['id_temporal'] = df['nombre'].astype(str) + " | " + df['unidad_de_medida'].astype(str)
    
    df_limpio = df[~df['id_temporal'].isin(ids_con_error)].copy()
    df_limpio = df_limpio.drop(columns=['id_temporal'])
    df = df_limpio
    
    print(f"Se detectaron {reporte_anomalias['total_anomalias']} variaciones sospechosas.")
    print(f"Se excluyeron {len(ids_con_error)} combinaciones de Producto+Unidad.")
    print(f"El analisis continuará con {len(df)} registros validados.")
else:
    print("No se detectaron anomalías significativas.")


In [ ]:
# Busca un producto que estuviera en el rango de "Datos Moderados" (15-30%)
comprobacion = stats_inicial[(stats_inicial['null_percentage'] > 15) & 
                        (stats_inicial['null_percentage'] <= 30)].head(1)

if not comprobacion.empty:
    nombre_prod = comprobacion['nombre'].values[0]
    
    print(f"\n{'='*60}")
    print(f"Comprobando imputación para: {nombre_prod}")
    print(f"{'='*60}")
    
    # Mostrar estadísticas del producto
    pct_nulos = comprobacion['null_percentage'].values[0]
    print(f"\nPorcentaje de nulos original: {pct_nulos:.2f}%")
    print("   Estrategia aplicada: Interpolación temporal\n")
    
    # Mostrar ANTES de la imputación (usando la copia guardada)
    print("ANTES DE LA IMPUTACIÓN:")
    print("-" * 40)
    datos_antes = df_antes_imputacion[df_antes_imputacion['nombre'] == nombre_prod][cols_fecha]
    
    # Configurar pandas para mostrar todo
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print(datos_antes)
    
    # Contar nulos antes
    nulos_antes = datos_antes.isna().sum().sum()
    print(f"\nValores nulos antes: {nulos_antes}")
    
    # Mostrar DESPUÉS de la imputación
    print(f"\n{'='*60}")
    print(" DESPUÉS DE LA IMPUTACIÓN:")
    print("-" * 40)
    datos_despues = df[df['nombre'] == nombre_prod][cols_fecha]
    
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print(datos_despues)
    
    # Contar nulos después
    nulos_despues = datos_despues.isna().sum().sum()
    print(f"\nValores nulos después: {nulos_despues}")
    
    # Mostrar resumen
    print(f"\n{'='*60}")
    print("RESUMEN DE IMPUTACIÓN:")
    print(f"   • Nulos eliminados: {nulos_antes - nulos_despues}")
    print("   • Método: Interpolación temporal (forward/backward fill)")
    print(f"{'='*60}")
    
else:
    print("No hay productos en el rango de 15-30% para mostrar ejemplo.")

In [10]:
df = calculate_average_price(df)
print("Precios promedio calculados")
print("\nEjemplo de datos finales:")
print(df[['nombre', 'categoria', 'unidad_de_medida', 'precio_promedio']].head(10))

Precios promedio calculados

Ejemplo de datos finales:
                      nombre categoria unidad_de_medida  precio_promedio
0      Aceite de soya Crisol    Aceite     Galón 128 Oz       663.001125
1     Aceite de soya La Joya    Aceite     Galón 128 Oz       622.050642
2            Aceite el Gallo    Aceite     Galón 128 Oz       644.758301
3            Aceite Diamante    Aceite     Galón 128 Oz       684.597158
4            Aceite Manicero    Aceite     Galón 128 Oz       727.559678
5   Aceite Crisol mas canola    Aceite     Galón 128 Oz       639.870510
7       Aceite Wesson canola    Aceite     Galón 128 Oz       975.345167
8      Aceite de soya Crisol    Aceite      Galón 64 Oz       323.213251
9     Aceite de Soya La Joya    Aceite      Galón 64 Oz       293.675114
10           Aceite el Gallo    Aceite      Galón 64 Oz       305.903915


## 6. Guardar datos transformados

In [ ]:
df.to_csv('../data/processed/data_transformada.csv', index=False)
print("✓ Datos transformados guardados: data/processed/data_transformada.csv")